In [1]:
import streamlit as st
from openai import OpenAI

In [4]:
# ollama server connection
client = OpenAI(
    base_url="http://localhost:11434/v1",  # URL of the Ollama server
    api_key="dummy_key",  # Add a dummy value
)


In [6]:
model="gemma3:1b"

In [2]:
from openai import OpenAI

# already configured in your app
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="dummy_key",
)

def summarize_transactions(transactions, model="gemma3:1b"):
    """
    Summarize recent transactions using Ollama LLM via OpenAI API.
    
    Args:
        transactions (list[dict]): List of transactions like
            [{"type": "Expense", "category": "Food", "amount": 500, "user_request": "Dinner"}, ...]
        model (str): Ollama model name (default = gemma3:270m)

    Returns:
        str: Natural language summary
    """
    if not transactions:
        return "No recent transactions found to summarize."

    # Convert transactions into a readable text block
    transactions_text = "\n".join([
        f"- {t['type']} | {t['category']} | {t['amount']} | {t['user_request']}"
        for t in transactions
    ])

    # Ask Ollama model to summarize
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a financial assistant. Summarize the transactions into a short, clear financial report."},
            {"role": "user", "content": f"Here are my recent transactions:\n{transactions_text}\n\nSummarize them in a few bullet points and give me a short insight."}
        ]
    )
    return response.choices[0].message.content.strip()


In [21]:
# Dummy transactions for testing
recent_transactions = [
    {"type": "Expense", "category": "Food", "amount": 1200, "user_request": "Lunch at KFC"},
    {"type": "Expense", "category": "Transport", "amount": 800, "user_request": "Uber ride"},
    {"type": "Income", "category": "Salary", "amount": 50000, "user_request": "Monthly salary"},
    {"type": "Expense", "category": "Entertainment", "amount": 1500, "user_request": "Movie tickets"}
]

summary = summarize_transactions(recent_transactions)
print(summary)


Okay, here’s a summary of your recent transactions, presented in bullet points with a brief insight:

**Financial Report Summary**

*   **Income:** You received a significant increase in income of $50,000 this month, indicating a potentially stable financial position.
*   **Expenses:**
    *   **Food:** $1200 for lunch at KFC – This suggests a moderate spending on dining out.
    *   **Transport:** $800 for an Uber ride – This indicates you’re using a vehicle for transportation.
    *   **Entertainment:** $1500 for movie tickets – This represents a noticeable allocation for leisure activities.
*   **Net:**  Based on this breakdown, your total expenses are slightly higher than your income, leaving a potential surplus or deficit to be monitored.

**Insight:** You've built a comfortable financial base with a strong income stream. However, it’s worth reviewing your spending habits to ensure you’re staying within your budget and maximizing your overall financial health.  Consider reviewing 

In [8]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a financial assistant."},
        {"role": "user", "content": f"hi"}
    ]
)
response.choices[0].message.content.strip()

'Hi there! How can I help you today? 😊 \n\nDo you have any questions for me, or would you like to talk about your finances?'

In [31]:
import json

In [152]:
def classify_agent(user_message: str) -> str:
    """
    Uses Gemma 3 to decide if the message should use:
    - 'expense_categorizer' → call the expense agent
    - 'general' → let Gemma respond
    """
    system_prompt = f"""
    You are an agent selector designed to classify user requests into one of the available agents.
    You MUST respond with a single, valid JSON object, with absolutely no markdown, extra text, or conversation outside of the JSON.

    Available agents:
    - "expense_categorizer": Select this if the user is asking to categorize, classify, or analyze a financial transaction or expense.
    - "general": Select this for all other requests, including general conversation, questions, or advice.

    TOOL OUTPUT FORMAT:
    Output ONLY a single, valid JSON object (no markdown, no extra text).

    1.  **If "expense_categorizer" is selected:**
        * The "args" must contain the key **"return_query"**.
        * The value must be a **CONCISE SUMMARY** of the transaction extracted from the user input (e.g., remove classifying verbs like "categorize this").

    2.  **If "general" is selected:**
        * The "args" must contain the key **"return_query"**.
        * The value must be the **EXACT ORIGINAL USER INPUT**.You should not summarize or remove anything for this.

   
    EXAMPLES:

    User: categorize this car service rs 1000
    Response:
    {{
        "tool_name": "expense_categorizer",
        "args": {{
            "return_query": "car service rs 1000"
        }}
    }}

    User: please classify this transaction - lunch 500
    Response:
    {{
        "tool_name": "expense_categorizer",
        "args": {{
            "return_query": "lunch 500"
        }}
    }}

    User: how are you?
    Response:
    {{
        "tool_name": "general",
        "args": {{
            "return_query": "how are you?"
        }}
    }}

    User: give me a financial advice.
    Response:
    {{
        "tool_name": "general",
        "args": {{
            "return_query": "give me a financial advice."
        }}
    }}
    
    User: What are my expenses for last week?.
    Response:
    {{
        "tool_name": "general",
        "args": {{
            "return_query": "What are my expenses for last week?"
        }}
    }}
    
    User: What category does my grocery bill of $120 fall into?
    Response:
    {{
        "tool_name": "expense_categorizer",
        "args": {{
        "return_query": "grocery bill $120"
        }}
    }}
    
    User: Analyze this expense: gas $45 at the station
    Response:
    {{
        "tool_name": "expense_categorizer",
        "args": {{
        "return_query": "gas $45"
        }}
    }}
    
    User: Tell me a joke about money.
    Response:
    {{
        "tool_name": "general",
        "args": {{
        "return_query": "Tell me a joke about money."
        }}
    }}
    
    User: How do I budget my monthly expenses effectively?
    Response:
    {{
        "tool_name": "general",
        "args": {{
        "return_query": "How do I budget my monthly expenses effectively?"
        }}
    }}
    User: What is the definition of inflation?
    Response:
    {{
        "tool_name": "general",
        "args": {{
        "return_query": "What is the definition of inflation?"
        }}
    }}
    
    User: Classify: rent payment 1500 euros
    Response:
    {{
        "tool_name": "expense_categorizer",
        "args": {{
        "return_query": "rent payment 1500 euros"
        }}
    }}
    """

    # response = client.chat.completions.create(
    #     model=model,
    #     messages=[
    #         {"role": "system", "content": system_prompt.strip()},
    #         {"role": "user", "content": user_message}
    #     ],
    #     temperature=0.0,
    #     # max_tokens=10
    # )

    # return response.choices[0].message.content.strip().lower()
    response = client.chat.completions.create(
    model=model,
    # response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ],
    temperature=0.0,
    response_format={"type": "json_object"}
    )
    raw_content = response.choices[0].message.content.strip()

    # 1. Remove the opening markdown block marker
    if raw_content.startswith('```json'):
        content = raw_content.lstrip('`\n').lstrip('json\n')
    else:
        content = raw_content

    # 2. Remove the closing markdown block marker
    if content.endswith('```'):
        final_json_string = content.rstrip('`')
    else:
        final_json_string = content
        
    json_string = final_json_string
    
    try:
        result_dict = json.loads(json_string)
        
        # Extract the required information
        tool_name = result_dict.get("tool_name")
        return_query = result_dict.get("args", {}).get("return_query")
        
        if tool_name == "general": #make sure every time when it hit genaral it get tha same user input
            return_query = user_message
        
        # print(f"raw: {final_json_string}")
        # print(f"Agent to use: {tool_name}")
        # print(f"Original query: {return_query}")
        
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
    
    
    return tool_name,return_query

In [154]:
classify_agent("hi")


('general', 'hi')

In [155]:
classify_agent("classify my coffee expense of $5.50")

('expense_categorizer', 'coffee $5.50')

In [156]:
classify_agent("bought gas for 600 rupees")

('expense_categorizer', 'gas 600 rupees')

In [157]:
classify_agent("need to categorize this: Uber ride 15 EUR")

('expense_categorizer', 'Uber ride 15 EUR')

In [158]:
classify_agent("how should I classify a payment of 120 for rent?")

('expense_categorizer', 'rent payment 120')

In [159]:
classify_agent("categorize: new laptop, 999 USD")

('expense_categorizer', 'new laptop 999 USD')

In [160]:
classify_agent("What is the definition of inflation?")

('general', 'What is the definition of inflation?')

In [161]:
classify_agent("What's a good savings strategy?")

('general', "What's a good savings strategy?")

In [162]:
classify_agent("Tell me a financial joke.")

('general', 'Tell me a financial joke.')

In [163]:
classify_agent("I need help with my budget for next month.")

('general', 'I need help with my budget for next month.')

In [164]:
classify_agent("What are my expenses for last week?")

('general', 'What are my expenses for last week?')